<a href="https://colab.research.google.com/github/RJDom10/Curso-Machine-Learning/blob/main/notebooks/01_preprocesamiento/03_Outliers_Reduccion_y_Tecnicas_Avanzadas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Preprocesamiento Avanzado: Outliers, Reduccion de Datos y Feature Engineering

**Curso de Machine Learning** | Modulo 1: Preparacion de Datos

---

## Introduccion

En las sesiones previas establecimos los fundamentos de la limpieza, codificacion, imputacion e integracion de datos. Sin embargo, para construir modelos robustos en Machine Learning, es imperativo abordar fenomenos mas complejos presentes en los datos del mundo real.

Este cuaderno aborda las ultimas etapas del preprocesamiento estandar y expande el conocimiento hacia tecnicas fundamentales en la frontera del modelado:
1. **Identificacion y Manejo de Outliers:** Distinguir entre errores y casos atipicos valiosos.
2. **Reduccion de Dimensionalidad:** Sintetizar la informacion preservando la varianza.
3. **Manejo de Clases Desbalanceadas (Tema Nuevo):** Tecnicas para evitar el sesgo algoritmico en clasificacion.
4. **Ingenieria de Caracteristicas (Tema Nuevo):** Creacion de variables polinomiales e interacciones para incrementar la expresividad algoritmica.

In [ ]:
# Importacion de librerias base para analisis de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuracion de estilo visual
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

# Carga del dataset creditApproval
file_path = '../../data/creditApproval_dataset.csv'

try:
    df = pd.read_csv(file_path)
    print("Dataset cargado exitosamente.")
except FileNotFoundError:
    print(f"No se encontro el archivo en la ruta {file_path}. Asegurese de que el archivo existe.")


---

## 1. Identificacion de Datos Anomalos (Outliers)

El objetivo principal de esta fase es detectar errores aleatorios o variaciones extremas en una variable medida. Nos referimos a este proceso como **identificacion** mas que como **eliminacion**, dado que un valor atipico no necesariamente es un error; a menudo contiene informacion critica (por ejemplo, en deteccion de fraudes).

Existen multiples enfoques para la deteccion de outliers. A continuacion, se exponen dos metodos estadisticos clasicos y un metodo algoritmico.

### 1.1 Metodo de Rango Intercuartilico (IQR)

El IQR mide la dispersion estadistica y es menos sensible a los valores extremos que la varianza. Se define como la diferencia entre el tercer y el primer cuartil ($IQR = Q3 - Q1$). Valores fuera del rango $[Q1 - 1.5 \times IQR, Q3 + 1.5 \times IQR]$ se consideran outliers.

In [ ]:
# Convertir a numerico ignorando errores para analizar la distribucion de 'A3'
df['A3'] = pd.to_numeric(df['A3'], errors='coerce')
data_a3 = df['A3'].dropna()

# Calculo de limites IQR
Q1 = data_a3.quantile(0.25)
Q3 = data_a3.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_iqr = data_a3[(data_a3 < lower_bound) | (data_a3 > upper_bound)]
print(f"Limites IQR: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Cantidad de Outliers detectados por IQR: {len(outliers_iqr)} ({(len(outliers_iqr)/len(data_a3)*100):.2f}%)")

# Visualizacion
plt.figure(figsize=(10, 4))
sns.boxplot(x=data_a3)
plt.title('Deteccion de Outliers en A3 mediante Boxplot (IQR)')
plt.show()

### 1.2 Deteccion Avanzada: Isolation Forest

El IQR es univariado (analiza una variable a la vez). En Machine Learning, frecuentemente lidiamos con outliers multivariados. Algoritmos como **Isolation Forest** aislan anomalias particionando el espacio de caracteristicas de forma aleatoria.

In [ ]:
from sklearn.ensemble import IsolationForest

# Preparar un subconjunto bivariado numerico (A3 y A8)
df['A8'] = pd.to_numeric(df['A8'], errors='coerce')
data_multi = df[['A3', 'A8']].dropna()

# Inicializar y ajustar el modelo
# contamination define la proporcion esperada de outliers
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outlier_preds = iso_forest.fit_predict(data_multi)

# Isolation Forest devuelve -1 para outliers y 1 para inliers
data_multi['Is_Outlier'] = np.where(outlier_preds == -1, 'Outlier', 'Inlier')

# Visualizacion del espacio multivariado
plt.figure(figsize=(8, 6))
sns.scatterplot(data=data_multi, x='A3', y='A8', hue='Is_Outlier', palette={'Inlier': 'blue', 'Outlier': 'red'})
plt.title('Deteccion Multivariada de Outliers usando Isolation Forest')
plt.show()

---

## 2. Reduccion de Datos y Dimensionalidad

La **reduccion de datos** comprende el conjunto de tecnicas que obtienen una representacion reducida de los datos originales, manteniendo la integridad de la informacion.

El objetivo es encontrar un subconjunto $C'$ que se aproxime al conjunto original $C$ ($C' \approx C$), conservando su varianza pero disminuyendo significativamente su volumen o dimensionalidad. Esto mitiga la maldicion de la dimensionalidad y reduce el coste computacional.

### Analisis de Componentes Principales (PCA)

PCA es un metodo estadistico que transforma variables posiblemente correlacionadas en un numero menor de variables no correlacionadas llamadas componentes principales. El primer componente explica la mayor varianza posible, el segundo la segunda mayor, y asi sucesivamente.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Preparacion: Extraer columnas numericas y tratar nulos
numeric_df = df.select_dtypes(include=[np.number]).dropna(axis=1, how='all')
imputer = SimpleImputer(strategy='median')
scaled_data = StandardScaler().fit_transform(imputer.fit_transform(numeric_df))

# Aplicar PCA para retener el 95% de la varianza
pca = PCA(n_components=0.95)
data_pca = pca.fit_transform(scaled_data)

print(f"Dimension original: {scaled_data.shape[1]} variables")
print(f"Dimension reducida: {data_pca.shape[1]} componentes principales")
print(f"Varianza explicada por componente: {pca.explained_variance_ratio_.round(3)}")
print(f"Varianza total acumulada: {sum(pca.explained_variance_ratio_):.3f}")

# Visualizacion de los dos primeros componentes
plt.figure(figsize=(8, 6))
plt.scatter(data_pca[:, 0], data_pca[:, 1], alpha=0.6, edgecolor='k')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('Proyeccion de Datos en los 2 Primeros Componentes Principales')
plt.show()

---

## 3. Manejo de Clases Desbalanceadas

En multiples escenarios practicos (fraude bancario, diagnostico medico), la variable objetivo (`class` o etiqueta) presenta un fuerte desequilibrio. Es decir, una clase es mayoritaria y la otra minoritaria.

Si alimentamos a un modelo con datos desbalanceados, el algoritmo tendera a predecir siempre la clase mayoritaria (alta exactitud, nula utilidad).

### Metodos de Remuestreo (Resampling)
* **Undersampling:** Reducir aleatoriamente muestras de la clase mayoritaria.
* **Oversampling:** Duplicar muestras de la clase minoritaria.
* **SMOTE (Synthetic Minority Over-sampling Technique):** Generar muestras sinteticas interpolando entre casos existentes de la clase minoritaria (Metodo preferido).

*Nota: La aplicacion de estas tecnicas requiere librerias externas como `imbalanced-learn`. Dado que el entorno esta enfocado en dependencias base, conceptualizamos su importancia aqui, ya que es el estandar industrial previo al entrenamiento.*

In [ ]:
# Analisis del balance de la variable objetivo 'class' en nuestro dataset
if 'class' in df.columns:
    class_counts = df['class'].value_counts()
    class_percentages = df['class'].value_counts(normalize=True) * 100
    
    summary = pd.DataFrame({
        'Conteo': class_counts,
        'Porcentaje (%)': class_percentages.round(2)
    })
    
    print("Distribucion de la variable objetivo:")
    display(summary)
    
    # Grafica de barras
    plt.figure(figsize=(6, 4))
    sns.barplot(x=class_counts.index, y=class_counts.values)
    plt.title('Balance de Clases en el Dataset')
    plt.ylabel('Numero de Observaciones')
    plt.xlabel('Clase')
    plt.show()
    
    print("Nota: En este caso, el balance (aprox. 55% vs 45%) es bastante saludable. ")
    print("Si tuvieramos una distribucion de 95% vs 5%, requeririamos usar tecnicas como SMOTE.")


---

## 4. Ingenieria de Caracteristicas (Feature Engineering)

La creacion de variables (Feature Engineering) es un arte fundamental en Machine Learning. Consiste en transformar o combinar variables existentes para crear nuevas variables que capturen mejor las relaciones subyacentes del problema.

Modelos lineales (como Regresion Logistica) asumen aditividad. Si el problema tiene comportamientos no lineales, podemos ayudar al modelo creando variables polinomicas o de interaccion.

### Polynomial Features e Interacciones

Transformaremos un subconjunto de caracteristicas a polinomios de grado 2, lo cual generara interacciones de la forma $A \times B$ asi como $A^2$ y $B^2$.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Tomaremos dos variables numericas para demostrar la creacion de atributos
# Utilizaremos las variables ya limpias e imputadas del paso PCA
X_demo = scaled_data[:, :2] # Supongamos que son A3 y A8

print(f"Dimension original de caracteristicas de prueba: {X_demo.shape[1]} (X1, X2)")

# Generar interacciones y polinomios de grado 2
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_demo)

print(f"Dimension tras PolynomialFeatures: {X_poly.shape[1]}")
print("Las nuevas caracteristicas representan: [X1, X2, X1^2, X1*X2, X2^2]")

# Muestra comparativa
print("\nObservacion 1 (Original):", X_demo[0].round(3))
print("Observacion 1 (Polinomial):", X_poly[0].round(3))

---

## Resumen del Modulo de Preprocesamiento

Concluimos la seccion de Preprocesamiento de Datos. El flujo de trabajo estandar en la industria comprende:

1. **Adquisicion y Exploracion:** Cargar y entender la semantica de cada variable.
2. **Limpieza:** Filtrar anomalias criticas y tratar valores nulos.
3. **Transformacion (Encoding):** Volver comprensible la informacion a un algoritmo matematico.
4. **Integracion:** Consolidar origenes de datos y agregaciones (Tablas Pivote).
5. **Deteccion de Outliers y Escalamiento:** Robustecer y normalizar la entrada numerica.
6. **Feature Engineering y Reduccion:** Sintetizar la informacion y crear descriptores ricos.
7. **Balanceo:** Asegurar equidad estadistica pre-entrenamiento.

Aplicar correctamente esta metodologia garantiza que la optimizacion y entrenamiento de nuestros modelos (Modulo 2) inicie sobre cimientos solidos.